<a href="https://colab.research.google.com/github/qqqweee/spatiotemporal_demand_forecasting/blob/main/spatiotemporal_demand_forecasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Spatiotemporal Demand Forecasting System

## Objective
Predict demand across geospatial regions using temporal and road-network features.

## Results
- 77K+ records
- 1,249 geospatial regions
- CatBoost Regressor
- R² Score: 0.86+

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score

from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

import optuna
import pygeohash

print("Setup Complete")

In [ ]:
import pandas as pd

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
sample = pd.read_csv("sample_submission.csv")
print("Train Shape:", train.shape)
print("Test Shape:", test.shape)
print("Sample Shape:", sample.shape)

print("\nTrain Columns:")
print(train.columns.tolist())

print("\nTest Columns:")
print(test.columns.tolist())

In [ ]:
train["RoadType"] = train["RoadType"].fillna("Unknown")
test["RoadType"] = test["RoadType"].fillna("Unknown")

train["Weather"] = train["Weather"].fillna("Unknown")
test["Weather"] = test["Weather"].fillna("Unknown")

median_temp = train["Temperature"].median()

train["Temperature"] = train["Temperature"].fillna(median_temp)
test["Temperature"] = test["Temperature"].fillna(median_temp)

In [ ]:
train[["hour", "minute"]] = (
    train["timestamp"]
    .str.split(":", expand=True)
    .astype(int)
)

test[["hour", "minute"]] = (
    test["timestamp"]
    .str.split(":", expand=True)
    .astype(int)
)

train["time_minutes"] = (
    train["hour"] * 60 +
    train["minute"]
)

test["time_minutes"] = (
    test["hour"] * 60 +
    test["minute"]
)

In [ ]:
import numpy as np

train["sin_time"] = np.sin(
    2 * np.pi * train["time_minutes"] / 1440
)

train["cos_time"] = np.cos(
    2 * np.pi * train["time_minutes"] / 1440
)

test["sin_time"] = np.sin(
    2 * np.pi * test["time_minutes"] / 1440
)

test["cos_time"] = np.cos(
    2 * np.pi * test["time_minutes"] / 1440
)

In [ ]:
import pygeohash as pgh
import pandas as pd

def decode_geohash(gh):
    lat, lon = pgh.decode(gh)
    return pd.Series([lat, lon])

train[["latitude", "longitude"]] = (
    train["geohash"]
    .apply(decode_geohash)
)

test[["latitude", "longitude"]] = (
    test["geohash"]
    .apply(decode_geohash)
)

In [ ]:
train["RoadType_LargeVehicles"] = (
    train["RoadType"].astype(str)
    + "_"
    + train["LargeVehicles"].astype(str)
)

test["RoadType_LargeVehicles"] = (
    test["RoadType"].astype(str)
    + "_"
    + test["LargeVehicles"].astype(str)
)

In [ ]:
train_df = train[train["day"] == 48].copy()
valid_df = train[train["day"] == 49].copy()
FEATURES = [
    "geohash",
    "RoadType",
    "NumberofLanes",
    "LargeVehicles",
    "sin_time",
    "RoadType_LargeVehicles"
]

CAT_COLS = [
    "geohash",
    "RoadType",
    "LargeVehicles",
    "RoadType_LargeVehicles"
]

X_train = train_df[FEATURES]
y_train = train_df["demand"]

X_valid = valid_df[FEATURES]
y_valid = valid_df["demand"]

print(X_train.shape)
print(X_valid.shape)

In [ ]:
from catboost import CatBoostRegressor

model = CatBoostRegressor(
    iterations=4000,
    depth=3,
    learning_rate=0.03,
    l2_leaf_reg=5,
    random_strength=1,
    loss_function="RMSE",
    eval_metric="R2",
    random_seed=42,
    verbose=200
)

model.fit(
    X_train,
    y_train,
    cat_features=CAT_COLS,
    eval_set=(X_valid, y_valid),
    use_best_model=True
)

In [ ]:
from sklearn.metrics import r2_score

preds = model.predict(X_valid)

print("R2 Score:", r2_score(y_valid, preds))

In [ ]:
fi = pd.DataFrame({
    "feature": FEATURES,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

print(fi)
print(model.get_best_iteration())